# PentaByte — Flower Prediction Explorer

**Unit V · Chapter 18 | Companion notebook for Detailed Markdown V2.0**

This is the same Iris classifier project as Units III–IV. Run all code cells in order. It uses scikit-learn's bundled Iris dataset, a training-only scaler, one eight-neuron MLP, and a 120/30 held-out split. The classifier and test set are not tuned using the final results.

**Learning goals:** Store prediction records, search by ID, sort by feature, compare model accuracy with a majority-class baseline, and explain a limitation.

**Setup:** Python 3 with `scikit-learn` installed. Use your institution-approved installation method if necessary.

**Diagram:** `Measurements → trained MLP → prediction → list of dictionaries → search / sort → display`.

**AI-use note:** This notebook was prepared with AI assistance and must be reviewed and run by the team before submission.

## 1. Load and split Iris

120 training and 30 held-out samples, with class proportions preserved.

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X = iris.data
Y = iris.target
class_names = iris.target_names
feature_names = iris.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y,
)

print("Classes:", list(class_names))
print("Training samples:", len(X_train))
print("Held-out samples:", len(X_test))

Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Training samples: 120
Held-out samples: 30


## 2. Train the original Unit IV MLP

Scaling is fitted only on training data through a pipeline.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(8,),
        activation="relu",
        solver="adam",
        learning_rate_init=0.01,
        alpha=0.01,
        max_iter=1200,
        random_state=42,
    ),
)

model.fit(X_train, y_train)
print("Trained MLP ready.")

Trained MLP ready.


## 3. Evaluate predictions and baseline

Use the same held-out split for both methods. Do not tune using test results.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

mlp_accuracy = accuracy_score(y_test, y_pred)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("MLP accuracy:", round(mlp_accuracy, 3))
print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Confusion matrix (actual rows, predicted columns):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))

MLP accuracy: 0.967
Baseline accuracy: 0.333
Confusion matrix (actual rows, predicted columns):
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


## 4. Build prediction records

Each record contains a local test-split ID, four measurements, true class, and predicted class.

In [4]:
records = []

for i, (measurements, actual, predicted) in enumerate(
    zip(X_test, y_test, y_pred), start=1
):
    records.append({
        "sample_id": f"T{i:03d}",
        "measurements": measurements.tolist(),
        "actual_label": int(actual),
        "predicted_label": int(predicted),
    })

print("Number of prediction records:", len(records))

for record in records[:3]:
    print(
        record["sample_id"],
        "actual:", class_names[record["actual_label"]],
        "predicted:", class_names[record["predicted_label"]],
        "petal width (cm):", record["measurements"][3],
    )

Number of prediction records: 30
T001 actual: setosa predicted: setosa petal width (cm): 0.2
T002 actual: virginica predicted: virginica petal width (cm): 1.8
T003 actual: versicolor predicted: versicolor petal width (cm): 1.0


## 5. Search by sample ID

A linear scan returns a dictionary for a match or None otherwise.

In [5]:
def find_record(records, requested_id):
    """Return the matching dictionary, or None if the ID is unknown."""
    for record in records:
        if record["sample_id"] == requested_id:
            return record
    return None

found = find_record(records, "T001")
missing = find_record(records, "T999")

print("T001 found:", found is not None)
print("T999 found:", missing is not None)

if found is not None:
    print("T001 prediction:", class_names[found["predicted_label"]])

T001 found: True
T999 found: False
T001 prediction: setosa


## 6. Sort by measurement

Sort complete records, not measurements alone, to preserve data association.

In [6]:
def sort_by_feature(records, feature_index):
    """Return a new list, ordered by the selected measurement."""
    if feature_index not in range(4):
        raise ValueError("feature_index must be 0, 1, 2, or 3")

    return sorted(
        records,
        key=lambda record: record["measurements"][feature_index],
    )

# Feature index 2 means petal length (in centimeters).
sorted_records = sort_by_feature(records, 2)

print("Sorted by:", feature_names[2])
for record in sorted_records[:5]:
    print(record["sample_id"], record["measurements"][2])

Sorted by: petal length (cm)
T007 1.0
T017 1.2
T001 1.3
T005 1.3
T016 1.4


## 7. Verify the workflow

Assertions check records, lookup, ordering, and evaluation.

In [7]:
assert len(records) == len(X_test) == len(y_test) == len(y_pred) == 30
assert len({record["sample_id"] for record in records}) == len(records)
assert all(len(record["measurements"]) == 4 for record in records)

assert find_record(records, "T001") is not None
assert find_record(records, "T999") is None

petal_lengths = [record["measurements"][2] for record in sorted_records]
assert petal_lengths == sorted(petal_lengths)
assert {r["sample_id"] for r in sorted_records} == {
    r["sample_id"] for r in records
}

matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
assert matrix.sum() == len(y_test)
assert 0 <= mlp_accuracy <= 1
assert 0 <= baseline_accuracy <= 1

print("All record, search, sorting, and evaluation checks passed.")

All record, search, sorting, and evaluation checks passed.


## 8. Explore a different sort feature and one real prediction

Change the sort key without retraining the classifier.

In [8]:
# Reader challenge: change sorting from petal length (2)
# to petal width (3); keep the trained model unchanged.
by_petal_width = sort_by_feature(records, 3)

print("Sorted by:", feature_names[3])
for record in by_petal_width[:5]:
    print(record["sample_id"], record["measurements"][3])

# Find and explain one real record from this run.
sample = find_record(records, "T001")
if sample is not None:
    print("Sample ID:", sample["sample_id"])
    print("Measurements (cm):", sample["measurements"])
    print("Model prediction:", class_names[sample["predicted_label"]])
    print("Known actual class:", class_names[sample["actual_label"]])

# This check verifies that the predictions were not changed by sorting.
assert all(
    find_record(by_petal_width, r["sample_id"])["predicted_label"]
    == r["predicted_label"]
    for r in records
)

Sorted by: petal width (cm)
T001 0.2
T005 0.2
T007 0.2
T008 0.2
T015 0.2
Sample ID: T001
Measurements (cm): [4.4, 3.0, 1.3, 0.2]
Model prediction: setosa
Known actual class: setosa


## 9. Display a compact actual results table

Copy measured results rather than inventing model outputs.

In [9]:
print("ID   Sepal length  Petal width  Actual       Predicted")
for record in records[:5]:
    m = record["measurements"]
    actual_name = class_names[record["actual_label"]]
    predicted_name = class_names[record["predicted_label"]]
    print(
        f'{record["sample_id"]:<4} {m[0]:>12.1f} {m[3]:>12.1f}  '
        f'{actual_name:<12} {predicted_name}'
    )

ID   Sepal length  Petal width  Actual       Predicted
T001          4.4          0.2  setosa       setosa
T002          6.1          1.8  virginica    virginica
T003          4.9          1.0  versicolor   versicolor
T004          5.0          1.0  versicolor   versicolor
T005          4.4          0.2  setosa       setosa


## Review

1. Which lines represent data structures, ordinary algorithms, and connectionist AI?
2. Why is `T999` handled without an exception?
3. What changes when switching from petal length to petal width as the sort key?
4. Why does strong accuracy on 30 Iris test flowers not guarantee performance on unfamiliar real-world data?

**Attribution:** scikit-learn's bundled Iris dataset and `MLPClassifier`, `StandardScaler`, `DummyClassifier`, and metrics. See the accompanying Unit V Markdown document for source links and explanations.